# Latent-space analysis — Multinucleation VAE

Adapted from the XAI course notebook (Part A.2.7 → A.2.9) to **this** model:

| Course (MNIST)            | Here (myotubes)                                  |
|---------------------------|--------------------------------------------------|
| 2-D latent, plot μ₁/μ₂    | **z_dim=10** → reduce with UMAP / PCA            |
| digit labels 0–9          | **biological labels**: `condition`, `replicate`  |
| 1-channel image           | **2 image channels**: membrane (ch0), nuclei (ch1) |
| `model.encode/decode`     | `model.vae.encoder` / `model.vae.decoder`        |

Sections:
1. Setup & encode the chosen split
2. **Latent-space visualization** (UMAP, coloured by condition/replicate, centroids)
3. **Posterior vs prior** (per-dim μ distributions, active units)
4. **Sampling from posteriors** (reparameterization spread)
5. **Decision boundaries** (logreg on latent space + confusion matrix)
6. **Sampling / generation** (decode condition centroids, interpolation, arbitrary point)
7. **Reconstructions** (best / worst)

> Edit `CKPT` below to point at the version you want (start: version_12).

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
# Point CKPT at the run you want to inspect.
CKPT  = "outputs/checkpoints/version_12/last.ckpt"   # or best epoch=..ckpt
ZARR  = "/mnt/efs/dl_jrc/student_data/S-JS/multinucleation.zarr"
TABLE = "outputs/cell_table.csv"
SPLIT = "test"          # train | val | test | all
BATCH = 64
WORKERS = 4
LABEL_COLS = ["condition", "replicate"]   # categorical metadata to colour by
SEED = 42

In [ ]:
import sys, os, contextlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# make the repo's src/ importable when running from Joaquin'scripts/ or repo root
for p in [Path.cwd(), Path.cwd() / "src", Path.cwd().parent, Path.cwd().parent / "src"]:
    if (p / "darth_vaeder").is_dir():
        sys.path.insert(0, str(p)); break

from darth_vaeder.datamodules import MultinucDataModule
from darth_vaeder.datamodules.JS_zarr_datamodule import vae_collate
from darth_vaeder.models import LitVAE
from torch.utils.data import ConcatDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@contextlib.contextmanager
def silence():                       # mute leftover debug print()s in vae.forward
    with open(os.devnull, "w") as dn, contextlib.redirect_stdout(dn):
        yield

print("device:", device)

In [ ]:
# ── Load model ──────────────────────────────────────────────────────────────
model = LitVAE.load_from_checkpoint(CKPT, map_location=device).eval().to(device)
h = model.hparams
print(f"z_dim={h.z_dim}  nc={h.nc}  nc_img={model.nc_img}  beta={h.beta}")
print(f"image_key={h.image_key}  mask_key={h.mask_key}  nuc_mask_key={h.nuc_mask_key}")

In [ ]:
# ── Dataloader (no augmentation: we want each cell's TRUE latent) ────────────
dm = MultinucDataModule(data_path=ZARR, cell_table_csv=TABLE, channels=(0, 1),
                        batch_size=BATCH, num_workers=WORKERS, augment=False)
dm.setup()
ds = {"train": dm.train_dataset, "val": dm.val_dataset, "test": dm.test_dataset}
dataset = ConcatDataset(list(ds.values())) if SPLIT == "all" else ds[SPLIT]
loader  = DataLoader(dataset, batch_size=BATCH, shuffle=False,
                     num_workers=WORKERS, collate_fn=vae_collate, drop_last=False)
print(f"split='{SPLIT}'  cells={len(dataset)}")

In [ ]:
# ── Encode the whole split → mu, logvar, recon error, metadata ──────────────
# Mirrors course `get_latent_features`, but uses the 4-channel encoder input
# (cnPatches + pCellmask + pNucmask) and the encoder directly (deterministic mu).
def build_x_in(batch):
    x_img    = batch[h.image_key].to(device)
    mask     = batch[h.mask_key].to(device)
    nuc_mask = batch[h.nuc_mask_key].to(device)
    return torch.cat([x_img, mask.float(), nuc_mask.float()], dim=1), x_img, mask

@torch.no_grad()
def get_latent_features(loader):
    mus, logvars, errs, meta_rows = [], [], [], []
    for bi, batch in enumerate(loader):
        x_in, x_img, mask = build_x_in(batch)
        with silence():
            mu, logvar = model.vae.encoder(x_in)
            recon = model.vae.decoder(mu)            # decode from mu (no noise)
        m2  = (mask > 0).expand_as(x_img).float()
        err = ((recon[:, :model.nc_img] - x_img) ** 2 * m2).sum([1,2,3]) / m2.sum([1,2,3]).clamp_min(1)
        mus.append(mu.cpu()); logvars.append(logvar.cpu()); errs.append(err.cpu())
        md = batch["metadata"]
        for j in range(x_img.shape[0]):
            meta_rows.append({k: md[k][j] for k in md})
        if (bi+1) % 10 == 0 or bi+1 == len(loader):
            print(f"  encoded {len(meta_rows)} cells")
    mus     = torch.cat(mus);   logvars = torch.cat(logvars)
    recon_err = torch.cat(errs).numpy()
    meta = pd.DataFrame(meta_rows); meta["recon_err"] = recon_err
    return mus, logvars, recon_err, meta

mus, logvars, recon_err, meta = get_latent_features(loader)
print("mu:", tuple(mus.shape), " mean recon_err:", round(float(recon_err.mean()), 4))
meta.head()

In [ ]:
# ── Per-class centroids (course `mu_mean`) — one mean latent per category ────
def class_means(mus, labels):
    cats = sorted(pd.unique(labels))
    means = {c: mus[np.asarray(labels) == c].mean(0) for c in cats}
    return cats, means

print("conditions:", sorted(meta["condition"].unique()))
print("replicates:", sorted(meta["replicate"].unique()))

## 2 · Latent-space visualization

z is 10-D, so we project to 2-D with **UMAP** (falls back to PCA if umap-learn is
missing). Big **X** markers are per-class centroids. The course question applies
directly: *do cells cluster by biological condition, or does the model just
memorise the replicate (batch effect)?*

In [ ]:
def reduce_2d(Z, method="umap", seed=SEED, extra=None):
    # Reduce to 2-D. If `extra` is given, embed it in the SAME space.
    X = Z if extra is None else np.concatenate([Z, extra])
    if method == "umap":
        try:
            from umap import UMAP
            emb = UMAP(n_components=2, random_state=seed).fit_transform(X)
            name = "UMAP"
        except ImportError:
            print("[warn] umap-learn missing → PCA"); method = "pca"
    if method == "pca":
        from sklearn.decomposition import PCA
        emb = PCA(n_components=2, random_state=seed).fit_transform(X); name = "PCA"
    if extra is None:
        return emb, name
    return emb[:len(Z)], emb[len(Z):], name

def scatter_by(ax, emb, labels, centroids_emb=None, title="", cmap="tab10", alpha=0.6, s=5):
    cats = sorted(pd.unique(labels)); cm = plt.get_cmap(cmap)
    lab = np.asarray(labels)
    for i, c in enumerate(cats):
        m = lab == c
        ax.scatter(emb[m, 0], emb[m, 1], s=s, color=cm(i), label=str(c),
                   alpha=alpha, linewidths=0, rasterized=True)
    if centroids_emb is not None:
        for i, c in enumerate(cats):
            ax.scatter(*centroids_emb[i], s=160, color="white", marker="X", zorder=9)
            ax.scatter(*centroids_emb[i], s=110, color=cm(i), edgecolors="black",
                       linewidths=0.6, marker="X", zorder=10)
    ax.legend(title="", markerscale=3, fontsize=8, loc="best")
    ax.set_title(title); ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
    ax.set_aspect("equal", adjustable="datalim")

Z = mus.numpy()

# embed cells + condition centroids together so centroids land in the same space
cond_cats, cond_means = class_means(mus, meta["condition"])
cent = np.stack([cond_means[c].numpy() for c in cond_cats])
emb, cent_emb, red_name = reduce_2d(Z, "umap", extra=cent)

fig, axes = plt.subplots(1, len(LABEL_COLS), figsize=(7*len(LABEL_COLS), 6), squeeze=False)
for ax, col in zip(axes[0], LABEL_COLS):
    ce = cent_emb if col == "condition" else None
    scatter_by(ax, emb, meta[col], centroids_emb=ce, title=f"{red_name} — {col}")
fig.suptitle(f"Latent space ({red_name}) — {SPLIT} split, n={len(Z)}", fontsize=13)
plt.tight_layout(); plt.show()

## 3 · Posterior vs prior

The KL term pulls each posterior `q(z|x)=N(μ,σ²)` toward the prior `N(0,1)`.
With **β=0** the latent is unconstrained (μ can drift far, σ tiny); with **β>0**
μ should concentrate near 0 and σ near 1. Below: per-dimension μ histograms
overlaid on the standard normal, plus the **active-units** bar (Var(μ) per dim —
dims with variance ≈0 are unused / collapsed).

In [ ]:
z_dim = Z.shape[1]
sigma = torch.exp(0.5 * logvars).numpy()       # posterior std per cell per dim

# per-dim mu histograms vs N(0,1)
ncol = 5; nrow = int(np.ceil(z_dim / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3*ncol, 2.2*nrow))
xs = np.linspace(-4, 4, 200); npdf = np.exp(-xs**2/2)/np.sqrt(2*np.pi)
for d in range(nrow*ncol):
    ax = axes.flat[d]
    if d >= z_dim: ax.axis("off"); continue
    ax.hist(Z[:, d], bins=60, density=True, color="steelblue", alpha=0.7)
    ax.plot(xs, npdf, "r--", lw=1, label="N(0,1)")
    ax.set_title(f"μ dim {d}  (σ̄={sigma[:,d].mean():.2f})", fontsize=8)
    ax.tick_params(labelsize=7)
axes.flat[0].legend(fontsize=7)
fig.suptitle("Per-dimension μ distribution vs N(0,1) prior", fontsize=12)
plt.tight_layout(); plt.show()

# active units
fig, ax = plt.subplots(figsize=(8, 3.5))
var = Z.var(0); order = np.argsort(var)[::-1]
ax.bar(range(z_dim), var[order], color="steelblue")
ax.set_xticks(range(z_dim)); ax.set_xticklabels([f"z{d}" for d in order], fontsize=8)
ax.set_ylabel("Var(μ)"); ax.set_title("Active units — variance of μ per latent dim")
plt.tight_layout(); plt.show()
print("active dims (var>0.1):", int((var > 0.1).sum()), "/", z_dim)

## 4 · Sampling from the posteriors

Course `sample_from_latents`: draw `z = μ + ε·σ` (reparameterization) many times
per cell and see how much the posteriors spread / overlap. Tight, well-separated
clouds per condition = a confident, structured latent.

In [ ]:
def sample_from_latents(mus, logvars, n_samples=30):
    std = torch.exp(0.5 * logvars).unsqueeze(1)            # (N,1,D)
    mu  = mus.unsqueeze(1)                                  # (N,1,D)
    z   = mu + torch.randn(mu.shape[0], n_samples, mu.shape[2]) * std
    return z.reshape(-1, mu.shape[2])                       # (N*n_samples, D)

n_samples = 30
z_samp = sample_from_latents(mus, logvars, n_samples).numpy()
lab_rep = np.repeat(meta["condition"].to_numpy(), n_samples)

# project samples into the SAME UMAP space is expensive; refit on a subsample
idx = np.random.default_rng(SEED).choice(len(z_samp), size=min(8000, len(z_samp)), replace=False)
emb_s, _ = reduce_2d(z_samp[idx], "umap")
fig, ax = plt.subplots(figsize=(7, 6))
scatter_by(ax, emb_s, lab_rep[idx], title=f"{n_samples} posterior samples / cell", alpha=0.15, s=3)
plt.tight_layout(); plt.show()

## 5 · Decision boundaries & classification

Course A.2.7.3: fit a logistic regression on the latent space and see how
separable the classes are. Two views:

* **Accuracy + confusion matrix** — fit on the **full 10-D** μ (the honest measure
  of how linearly separable the conditions are in latent space).
* **Decision-boundary plot** — a 2nd logreg fit on the **2-D embedding** purely so
  the boundaries can be drawn (visualization only; lower-D so usually less accurate).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

TARGET = "condition"     # try "replicate" to probe batch effect
y = meta[TARGET].to_numpy()

# (a) full-D logistic regression
Xtr, Xte, ytr, yte = train_test_split(Z, y, test_size=0.25, random_state=SEED, stratify=y)
clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
pred = clf.predict(Xte)
acc = accuracy_score(yte, pred)
cats = sorted(pd.unique(y)); cm = confusion_matrix(yte, pred, labels=cats)
print(f"full-{Z.shape[1]}D logreg accuracy on {TARGET}: {acc:.3f}")

# (b) 2-D boundary logreg on the embedding
clf2 = LogisticRegression(max_iter=2000).fit(emb, y)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
# confusion matrix
ax = axes[0]
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats, rotation=45, ha="right")
ax.set_yticks(range(len(cats))); ax.set_yticklabels(cats)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion — {TARGET}\nfull-D acc={acc:.3f}")
for i in range(len(cats)):
    for j in range(len(cats)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=9)
# decision boundaries on embedding
ax = axes[1]
xmin, xmax = emb[:,0].min()-1, emb[:,0].max()+1
ymin, ymax = emb[:,1].min()-1, emb[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(xmin, xmax, 400), np.linspace(ymin, ymax, 400))
zz = clf2.predict(np.c_[xx.ravel(), yy.ravel()])
zz_idx = np.vectorize({c: i for i, c in enumerate(sorted(set(y)))}.get)(zz).reshape(xx.shape)
ax.contourf(xx, yy, zz_idx, alpha=0.2, cmap="tab10",
            levels=np.arange(-0.5, len(cats), 1))
scatter_by(ax, emb, y, title=f"Decision regions on {red_name} ({TARGET})")
plt.tight_layout(); plt.show()

## 6 · Sampling / generation from the latent space

### 6a · Decode the condition centroids
Course `gen_mean_numbers`: decode the mean latent of each class. This is the
model's "prototype" cell for each condition. Membrane (red) and nuclei (green)
shown separately and as an RGB composite.

In [ ]:
def decode(z):
    # z: (B, z_dim) -> (B, nc_img, H, W) numpy
    z = torch.as_tensor(z, dtype=torch.float32, device=device)
    if z.ndim == 1: z = z[None]
    with torch.no_grad(), silence():
        rec = model.vae.decoder(z)
    return rec[:, :model.nc_img].cpu().numpy()

def norm01(a):
    return (a - a.min()) / (a.ptp() + 1e-6)

def composite(img2):     # (2,H,W) → RGB (H,W,3): R=membrane, G=nuclei
    return np.stack([norm01(img2[0]), norm01(img2[1]), np.zeros_like(img2[0])], -1)

cats = cond_cats
dec = decode(np.stack([cond_means[c].numpy() for c in cats]))
fig, axes = plt.subplots(3, len(cats), figsize=(2.6*len(cats), 7.5))
for j, c in enumerate(cats):
    axes[0, j].imshow(norm01(dec[j, 0]), cmap="gray"); axes[0, j].set_title(str(c), fontsize=10)
    axes[1, j].imshow(norm01(dec[j, 1]), cmap="gray")
    axes[2, j].imshow(composite(dec[j]))
    for r in range(3): axes[r, j].axis("off")
for r, lbl in enumerate(["membrane", "nuclei", "RGB"]):
    axes[r, 0].set_ylabel(lbl, fontsize=10, rotation=90); axes[r, 0].axis("on")
    axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
fig.suptitle("Decoded condition centroids (prototypes)", fontsize=12)
plt.tight_layout(); plt.show()

### 6b · Interpolate between two condition centroids
Walk a straight line between two prototypes and decode each step — does the
morphology change smoothly (well-structured latent) or jump?

In [ ]:
COND_A = cats[0]
COND_B = cats[-1]
steps  = 10

za, zb = cond_means[COND_A].numpy(), cond_means[COND_B].numpy()
path = np.stack([(1-t)*za + t*zb for t in np.linspace(0, 1, steps)])
dec = decode(path)
fig, axes = plt.subplots(2, steps, figsize=(1.7*steps, 3.6))
for s in range(steps):
    axes[0, s].imshow(norm01(dec[s, 0]), cmap="gray"); axes[0, s].axis("off")
    axes[1, s].imshow(norm01(dec[s, 1]), cmap="gray"); axes[1, s].axis("off")
    axes[0, s].set_title(f"{1-s/(steps-1):.1f}/{s/(steps-1):.1f}", fontsize=7)
axes[0,0].set_ylabel("membrane", fontsize=9); axes[1,0].set_ylabel("nuclei", fontsize=9)
fig.suptitle(f"Interpolation  {COND_A} → {COND_B}", fontsize=12)
plt.tight_layout(); plt.show()

### 6c · Latent traversals
Course "pick a point in latent space and decode". Here we walk **each latent dim**
from −3σ to +3σ around the global mean, holding the others fixed — reveals what
morphological factor each axis controls (size, elongation, nuclei count, …).

In [ ]:
trav_steps = 7; span = 3.0
mu_mean = mus.mean(0); std = mus.std(0).numpy()
alphas = np.linspace(-span, span, trav_steps)
fig, axes = plt.subplots(z_dim, trav_steps, figsize=(1.4*trav_steps, 1.4*z_dim))
axes = np.atleast_2d(axes)
for d in range(z_dim):
    zs = mu_mean.repeat(trav_steps, 1).clone()
    zs[:, d] = mu_mean[d] + torch.tensor(alphas * std[d], dtype=torch.float32)
    dec = decode(zs.numpy())
    for s in range(trav_steps):
        axes[d, s].imshow(norm01(dec[s, 0]), cmap="gray"); axes[d, s].axis("off")
        if d == 0: axes[d, s].set_title(f"{alphas[s]:+.0f}σ", fontsize=8)
    axes[d, 0].set_ylabel(f"z{d}", fontsize=8, rotation=0, labelpad=12, va="center")
fig.suptitle("Latent traversals — membrane channel", fontsize=12)
plt.tight_layout(); plt.show()

## 7 · Reconstructions (best / worst)
Sanity check on the autoencoder: lowest- and highest-error cells in the split.

In [ ]:
n = 6
order = np.argsort(recon_err)
sets = {"best": order[:n], "worst": order[::-1][:n]}

# fetch the specific cells (shuffle-free second pass)
wanted = {int(i): None for grp in sets.values() for i in grp}
gi = 0
for batch in loader:
    x_img = batch[h.image_key]; mask = batch[h.mask_key]; nuc = batch[h.nuc_mask_key]
    for j in range(x_img.shape[0]):
        if gi + j in wanted:
            wanted[gi+j] = (x_img[j], mask[j], nuc[j])
    gi += x_img.shape[0]
    if all(v is not None for v in wanted.values()): break

fig, axes = plt.subplots(4, n, figsize=(1.8*n, 7.5))
row = 0
for name, idxs in sets.items():
    for k, idx in enumerate(idxs):
        x_img_i, mask_i, nuc_i = wanted[int(idx)]
        x_in = torch.cat([x_img_i, mask_i.float(), nuc_i.float()], 0)[None].to(device)
        with torch.no_grad(), silence():
            rec, *_ = model.vae(x_in)
        axes[row,   k].imshow(norm01(x_img_i[0].numpy()), cmap="gray"); axes[row,   k].axis("off")
        axes[row+1, k].imshow(norm01(rec[0,0].cpu().numpy()), cmap="gray"); axes[row+1, k].axis("off")
        axes[row, k].set_title(f"err={recon_err[idx]:.3f}", fontsize=7)
    axes[row,   0].set_ylabel(f"{name}\ninput", fontsize=8); axes[row,0].axis("on"); axes[row,0].set_xticks([]); axes[row,0].set_yticks([])
    axes[row+1, 0].set_ylabel("recon", fontsize=8); axes[row+1,0].axis("on"); axes[row+1,0].set_xticks([]); axes[row+1,0].set_yticks([])
    row += 2
fig.suptitle("Reconstructions — membrane channel", fontsize=12)
plt.tight_layout(); plt.show()